# AI Professor — Avaliação RAGAS
Métricas: `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`

Fluxo:
1. Carrega chunks do Qdrant
2. Claude gera perguntas + ground truths a partir dos chunks
3. RAG responde cada pergunta (retrieve + generate)
4. RAGAS avalia

In [ ]:
!pip install -q ragas==0.1.21 anthropic sentence-transformers 'qdrant-client[fastembed]' datasets langchain langchain-anthropic langchain-community

In [ ]:
from google.colab import userdata

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

COLLECTION = 'ai_professor_docs'
DENSE_MODEL = 'intfloat/multilingual-e5-large'
SPARSE_MODEL = 'Qdrant/bm25'
N_QUESTIONS = 10  # número de perguntas para avaliação

In [ ]:
# Carregar chunks do Qdrant
from qdrant_client import QdrantClient

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Scroll para pegar todos os chunks
all_points, _ = client.scroll(
    collection_name=COLLECTION,
    limit=1000,
    with_payload=True,
    with_vectors=False,
)
chunks = [p.payload for p in all_points]
print(f'Total de chunks carregados: {len(chunks)}')

In [ ]:
# Gerar perguntas + ground truths com Claude a partir de chunks amostrados
import anthropic
import random
import json

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Amostrar chunks distribuídos ao longo da aula
step = max(1, len(chunks) // N_QUESTIONS)
sampled = [chunks[i] for i in range(0, len(chunks), step)][:N_QUESTIONS]

test_questions = []
ground_truths = []

for chunk in sampled:
    prompt = f"""Com base no trecho de aula abaixo, gere:
1. Uma pergunta objetiva que pode ser respondida com o conteúdo do trecho
2. A resposta correta e completa para essa pergunta

Responda APENAS em JSON com as chaves "question" e "ground_truth".

Trecho:
{chunk['text'][:800]}"""

    response = claude.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}]
    )
    
    try:
        raw = response.content[0].text.strip()
        # Extrair JSON mesmo que venha com texto ao redor
        start = raw.index('{')
        end = raw.rindex('}') + 1
        data = json.loads(raw[start:end])
        test_questions.append(data['question'])
        ground_truths.append(data['ground_truth'])
        print(f'Q: {data["question"]}')
    except Exception as e:
        print(f'Erro ao parsear: {e} | Raw: {raw[:100]}')

print(f'\n{len(test_questions)} perguntas geradas.')

In [ ]:
# Inicializar modelos de retrieval
from sentence_transformers import SentenceTransformer
from fastembed.sparse.bm25 import Bm25
from qdrant_client.http.models import SparseVector
from collections import defaultdict

dense_model = SentenceTransformer(DENSE_MODEL)
bm25 = Bm25(SPARSE_MODEL)
print('Modelos de retrieval prontos.')

def retrieve(query, top_k=4):
    q_dense = dense_model.encode('query: ' + query, normalize_embeddings=True).tolist()
    q_sparse_raw = list(bm25.embed([query]))[0]
    q_sparse = SparseVector(
        indices=q_sparse_raw.indices.tolist(),
        values=q_sparse_raw.values.tolist()
    )

    dense_hits = client.query_points(
        collection_name=COLLECTION, query=q_dense, using='dense', limit=top_k * 3, with_payload=True
    ).points
    sparse_hits = client.query_points(
        collection_name=COLLECTION, query=q_sparse, using='sparse', limit=top_k * 3, with_payload=True
    ).points

    K = 60
    scores = defaultdict(float)
    payloads = {}
    for rank, hit in enumerate(dense_hits):
        sid = str(hit.id)
        scores[sid] += 1 / (K + rank + 1)
        payloads[sid] = hit.payload or {}
    for rank, hit in enumerate(sparse_hits):
        sid = str(hit.id)
        scores[sid] += 1 / (K + rank + 1)
        payloads[sid] = hit.payload or {}

    top_ids = sorted(scores, key=lambda x: scores[x], reverse=True)[:top_k]
    return [payloads[sid].get('text', '') for sid in top_ids]

In [ ]:
# Gerar respostas RAG para cada pergunta
answers = []
contexts = []

SYSTEM_PROMPT = """Você é um assistente educacional. Responda a pergunta do aluno 
usando APENAS as informações do contexto fornecido. 
Se o contexto não contiver a informação, diga que não encontrou na aula.
Seja objetivo e claro."""

for question in test_questions:
    retrieved = retrieve(question)
    contexts.append(retrieved)

    context_text = '\n\n'.join(retrieved)
    user_msg = f"Contexto da aula:\n{context_text}\n\nPergunta: {question}"

    response = claude.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_msg}]
    )
    answer = response.content[0].text.strip()
    answers.append(answer)
    print(f'Q: {question[:80]}...')
    print(f'A: {answer[:120]}...')
    print()

print(f'{len(answers)} respostas geradas.')

In [ ]:
# Avaliar com RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_anthropic import ChatAnthropic
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# LLM e embeddings para RAGAS
ragas_llm = LangchainLLMWrapper(
    ChatAnthropic(model='claude-haiku-4-5-20251001', api_key=ANTHROPIC_API_KEY)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=DENSE_MODEL)
)

# Configurar métricas com LLM customizado
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for metric in metrics:
    metric.llm = ragas_llm
    if hasattr(metric, 'embeddings'):
        metric.embeddings = ragas_embeddings

# Dataset RAGAS — ground_truths é lista de listas
dataset = Dataset.from_dict({
    'question': test_questions,
    'answer': answers,
    'contexts': contexts,
    'ground_truths': [[gt] for gt in ground_truths],
})

print('Iniciando avaliação RAGAS...')
result = evaluate(dataset, metrics=metrics)
print('\n=== Resultados RAGAS ===')
print(result)

In [ ]:
# Exibir resultados em tabela
import pandas as pd

df = result.to_pandas()

print('=== Médias por métrica ===')
summary = {
    'faithfulness': df['faithfulness'].mean(),
    'answer_relevancy': df['answer_relevancy'].mean(),
    'context_precision': df['context_precision'].mean(),
    'context_recall': df['context_recall'].mean(),
}
for metric, value in summary.items():
    status = '✅' if value >= 0.75 else '⚠️' if value >= 0.5 else '❌'
    print(f'{status} {metric:25s}: {value:.3f}')

print('\n=== Resultado por pergunta ===')
display(df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']])